# V7: Evaluation-Gated Skill Learning

V6 can remember reusable procedures across runs. V7 asks a harder question: **when should a remembered procedure become an operational Skill that changes future execution?**

This notebook implements a controlled learning loop:

`successful trajectories -> candidate Skill -> shadow evaluation -> deterministic gate -> versioned promotion -> selective reuse`

The agent never edits its own production prompt or activates a strategy merely because an LLM says it is useful. A Skill becomes active only after measured improvement on representative cases with no policy regression.

## 0. Source Map and Design Rationale

- [OpenAI Skills](https://developers.openai.com/api/docs/guides/tools-skills): describes a Skill as a versioned bundle of reusable instructions and resources that codifies a process or multi-step workflow.
- [OpenAI Agent Evals](https://developers.openai.com/api/docs/guides/agent-evals): motivates reproducible evaluation of agent behavior rather than judging one impressive trace.
- [Anthropic: Demystifying Evals for AI Agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents): treats multi-turn trajectories, tool use, state changes, and outcomes as evaluation targets.
- [Voyager](https://arxiv.org/abs/2305.16291): demonstrates an interpretable, compositional Skill library learned from environment feedback and reused on new tasks.

V7 adapts these ideas to a research agent while adding a strict safety boundary:

- memories may propose a Skill;
- shadow evaluation may measure it;
- deterministic policy decides promotion;
- only active versions may affect live routing;
- an untested revision never replaces the active version.

## 1. Setup

In [1]:
import json
import os
import re
from enum import Enum

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True))

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5-nano")
USE_LIVE_MODEL = os.getenv("V7_USE_LIVE_MODEL", "1") == "1"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None

print(f"[V7] Model: {MODEL_NAME}")
print(f"[V7] Live synthesis enabled: {USE_LIVE_MODEL and client is not None}")

[V7] Model: gpt-5-nano
[V7] Live synthesis enabled: True


## 2. Skill Contracts

A Skill is procedural memory promoted into a versioned execution policy. Its contract contains:

- **triggers and eligible categories** for selection;
- **steps** for execution;
- **invariants** that cannot be relaxed by the strategy;
- **provenance** linking it to prior memories and runs;
- **lifecycle status** separating candidate, shadow, active, rejected, and retired versions.

This is intentionally narrower than fine-tuning. The model weights do not change; the harness changes which tested procedure is injected for an eligible task.

In [2]:
class SkillStatus(str, Enum):
    CANDIDATE = "candidate"
    SHADOW = "shadow"
    ACTIVE = "active"
    REJECTED = "rejected"
    RETIRED = "retired"


class SkillStep(BaseModel):
    order: int = Field(ge=1)
    instruction: str = Field(min_length=8)


class SkillSpec(BaseModel):
    id: str
    name: str
    version: int = Field(ge=1)
    description: str
    trigger_terms: set[str]
    eligible_categories: set[str]
    steps: list[SkillStep]
    invariants: list[str]
    source_memory_ids: list[str]
    source_run_ids: list[str]
    status: SkillStatus = SkillStatus.CANDIDATE
    parent_skill_id: str | None = None


class BenchmarkCase(BaseModel):
    id: str
    category: str
    request: str
    required_evidence_lanes: set[str]


class CaseExecution(BaseModel):
    case_id: str
    skill_id: str | None
    skill_used: bool
    tool_calls: int
    quality_score: float
    evidence_coverage: float
    policy_violations: int


class SkillEvaluation(BaseModel):
    skill_id: str
    relevant_cases: int
    baseline_quality: float
    candidate_quality: float
    baseline_tool_calls: float
    candidate_tool_calls: float
    quality_delta: float
    tool_call_delta: float
    policy_violations: int
    ineligible_misroutes: int
    passed: bool
    case_results: list[CaseExecution]


class PromotionPolicy(BaseModel):
    minimum_relevant_cases: int = 3
    minimum_quality_delta: float = 0.10
    minimum_tool_call_reduction: float = 1.0
    maximum_policy_violations: int = 0
    maximum_ineligible_misroutes: int = 0

## 3. Distill a Candidate Skill from V6 Memory

The `StrategyLearner` reads compact, provenance-bearing memories rather than raw hidden reasoning. It produces a candidate that is inspectable but not executable in active traffic.

In a fully model-driven implementation, structured output could propose the fields. The lifecycle gate would remain deterministic because generation and authorization are different responsibilities.

In [3]:
class StrategyLearner:
    def distill(self, memories: list[dict[str, object]]) -> SkillSpec:
        procedural = [
            memory
            for memory in memories
            if memory["kind"] == "procedural"
        ]
        if not procedural:
            raise ValueError("At least one procedural memory is required.")

        memory_ids = [str(memory["id"]) for memory in memories]
        source_run_ids = sorted(
            {
                str(run_id)
                for memory in memories
                for run_id in memory["source_run_ids"]
            }
        )

        return SkillSpec(
            id="academic_first_research_v1",
            name="academic_first_research",
            version=1,
            description=(
                "Collect technical evidence with an academic-first query, "
                "triangulate with official documentation, deduplicate, and stop "
                "after all required evidence lanes are covered."
            ),
            trigger_terms={"research", "agent", "reliability", "evidence", "memory"},
            eligible_categories={"technical_research"},
            steps=[
                SkillStep(order=1, instruction="Normalize the research question and evidence lanes."),
                SkillStep(order=2, instruction="Search an academic index with precise technical terminology."),
                SkillStep(order=3, instruction="Triangulate key claims with first-party documentation."),
                SkillStep(order=4, instruction="Deduplicate sources and record claim-level provenance."),
                SkillStep(order=5, instruction="Stop when required evidence lanes and quality thresholds pass."),
            ],
            invariants=[
                "Never fabricate citations.",
                "Never bypass source provenance.",
                "Respect tool-call and latency budgets.",
            ],
            source_memory_ids=memory_ids,
            source_run_ids=source_run_ids,
        )


v6_memories = [
    {
        "id": "memory_1",
        "kind": "episodic",
        "content": "Broad exploration produced redundant evidence and four tool calls.",
        "source_run_ids": ["run_001"],
    },
    {
        "id": "memory_3",
        "kind": "procedural",
        "content": "Use academic-first triangulation and an evidence stopping rule.",
        "source_run_ids": ["run_001"],
    },
]

learner = StrategyLearner()
candidate_skill = learner.distill(v6_memories)

print(f"[V7][LEARNER] Candidate: {candidate_skill.id}")
print(f"[V7][LEARNER] Status: {candidate_skill.status.value}")
print(f"[V7][LEARNER] Provenance memories: {candidate_skill.source_memory_ids}")

[V7][LEARNER] Candidate: academic_first_research_v1
[V7][LEARNER] Status: candidate
[V7][LEARNER] Provenance memories: ['memory_1', 'memory_3']


## 4. Deterministic Skill Selection and Execution

Selection uses an explicit action mask:

1. the Skill must be active for live execution;
2. the task category must be eligible;
3. at least one trigger term must match;
4. evaluation can exercise a shadow Skill without making it globally active.

The controlled environment makes the causal comparison visible. A good Skill reduces redundant calls and improves evidence coverage. A deliberately bad “exhaustive search” Skill increases cost and violates the tool budget.

In [4]:
def token_set(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def skill_applies(skill: SkillSpec, case: BenchmarkCase) -> bool:
    return (
        case.category in skill.eligible_categories
        and bool(token_set(case.request) & skill.trigger_terms)
    )


def execute_case(
    case: BenchmarkCase,
    skill: SkillSpec | None = None,
    allow_shadow: bool = False,
) -> CaseExecution:
    eligible_status = skill and (
        skill.status == SkillStatus.ACTIVE
        or (allow_shadow and skill.status == SkillStatus.SHADOW)
    )
    use_skill = bool(eligible_status and skill_applies(skill, case))

    if case.category == "breaking_news":
        return CaseExecution(
            case_id=case.id,
            skill_id=None,
            skill_used=False,
            tool_calls=2,
            quality_score=0.91,
            evidence_coverage=1.0,
            policy_violations=0,
        )

    if use_skill and "exhaustive" in skill.name:
        return CaseExecution(
            case_id=case.id,
            skill_id=skill.id,
            skill_used=True,
            tool_calls=10,
            quality_score=0.86,
            evidence_coverage=1.0,
            policy_violations=1,
        )

    if use_skill:
        return CaseExecution(
            case_id=case.id,
            skill_id=skill.id,
            skill_used=True,
            tool_calls=2,
            quality_score=0.96,
            evidence_coverage=1.0,
            policy_violations=0,
        )

    return CaseExecution(
        case_id=case.id,
        skill_id=None,
        skill_used=False,
        tool_calls=4,
        quality_score=0.78,
        evidence_coverage=0.67,
        policy_violations=0,
    )

## 5. Skill Registry and Shadow Evaluation Gate

The registry owns lifecycle transitions. The Skill author cannot promote its own candidate.

Evaluation compares baseline and shadow behavior on the same cases. Promotion requires:

- enough relevant cases;
- a minimum quality improvement;
- fewer tool calls;
- zero policy violations;
- zero activation on ineligible tasks.

In [5]:
class SkillRegistry:
    def __init__(self, policy: PromotionPolicy | None = None):
        self.policy = policy or PromotionPolicy()
        self.skills: dict[str, SkillSpec] = {}
        self.evaluations: dict[str, SkillEvaluation] = {}

    def register(self, skill: SkillSpec) -> None:
        if skill.id in self.skills:
            raise ValueError(f"Skill {skill.id!r} already exists.")
        self.skills[skill.id] = skill.model_copy(deep=True)

    def begin_shadow(self, skill_id: str) -> SkillSpec:
        skill = self.skills[skill_id]
        if skill.status != SkillStatus.CANDIDATE:
            raise ValueError("Only candidate Skills can enter shadow evaluation.")
        skill.status = SkillStatus.SHADOW
        return skill

    def evaluate(
        self,
        skill_id: str,
        cases: list[BenchmarkCase],
    ) -> SkillEvaluation:
        skill = self.skills[skill_id]
        if skill.status != SkillStatus.SHADOW:
            raise ValueError("Skill must be in shadow status before evaluation.")

        baselines = [execute_case(case) for case in cases]
        candidates = [
            execute_case(case, skill=skill, allow_shadow=True)
            for case in cases
        ]
        relevant_indexes = [
            index
            for index, case in enumerate(cases)
            if skill_applies(skill, case)
        ]
        ineligible_indexes = [
            index for index in range(len(cases)) if index not in relevant_indexes
        ]

        def average(values: list[float]) -> float:
            return round(sum(values) / len(values), 3)

        baseline_quality = average(
            [baselines[index].quality_score for index in relevant_indexes]
        )
        candidate_quality = average(
            [candidates[index].quality_score for index in relevant_indexes]
        )
        baseline_calls = average(
            [float(baselines[index].tool_calls) for index in relevant_indexes]
        )
        candidate_calls = average(
            [float(candidates[index].tool_calls) for index in relevant_indexes]
        )
        policy_violations = sum(item.policy_violations for item in candidates)
        ineligible_misroutes = sum(
            candidates[index].skill_used for index in ineligible_indexes
        )
        quality_delta = round(candidate_quality - baseline_quality, 3)
        call_delta = round(baseline_calls - candidate_calls, 3)

        passed = all(
            [
                len(relevant_indexes) >= self.policy.minimum_relevant_cases,
                quality_delta >= self.policy.minimum_quality_delta,
                call_delta >= self.policy.minimum_tool_call_reduction,
                policy_violations <= self.policy.maximum_policy_violations,
                ineligible_misroutes <= self.policy.maximum_ineligible_misroutes,
            ]
        )

        evaluation = SkillEvaluation(
            skill_id=skill_id,
            relevant_cases=len(relevant_indexes),
            baseline_quality=baseline_quality,
            candidate_quality=candidate_quality,
            baseline_tool_calls=baseline_calls,
            candidate_tool_calls=candidate_calls,
            quality_delta=quality_delta,
            tool_call_delta=call_delta,
            policy_violations=policy_violations,
            ineligible_misroutes=ineligible_misroutes,
            passed=passed,
            case_results=candidates,
        )
        self.evaluations[skill_id] = evaluation
        return evaluation

    def decide_promotion(self, skill_id: str) -> SkillSpec:
        skill = self.skills[skill_id]
        evaluation = self.evaluations.get(skill_id)
        if evaluation is None:
            raise ValueError("A completed evaluation is required.")
        skill.status = (
            SkillStatus.ACTIVE if evaluation.passed else SkillStatus.REJECTED
        )
        return skill

    def retrieve(self, case: BenchmarkCase) -> SkillSpec | None:
        eligible = [
            skill
            for skill in self.skills.values()
            if skill.status == SkillStatus.ACTIVE and skill_applies(skill, case)
        ]
        if not eligible:
            return None
        return max(eligible, key=lambda item: item.version)

## 6. Evaluate a Good Skill and Reject a Bad Skill

In [6]:
benchmark_cases = [
    BenchmarkCase(
        id="case_1",
        category="technical_research",
        request="Research reliable agent memory with academic evidence.",
        required_evidence_lanes={"academic", "official"},
    ),
    BenchmarkCase(
        id="case_2",
        category="technical_research",
        request="Compare agent reliability patterns using source-backed research.",
        required_evidence_lanes={"academic", "official"},
    ),
    BenchmarkCase(
        id="case_3",
        category="technical_research",
        request="Research long-horizon agent evaluation methods.",
        required_evidence_lanes={"academic", "official"},
    ),
    BenchmarkCase(
        id="case_4_news",
        category="breaking_news",
        request="Summarize breaking market news from the last hour.",
        required_evidence_lanes={"fresh_news"},
    ),
]

registry = SkillRegistry()
registry.register(candidate_skill)
registry.begin_shadow(candidate_skill.id)
good_evaluation = registry.evaluate(candidate_skill.id, benchmark_cases)
promoted_skill = registry.decide_promotion(candidate_skill.id)

bad_skill = SkillSpec(
    id="exhaustive_search_v1",
    name="exhaustive_search",
    version=1,
    description="Search repeatedly regardless of evidence coverage.",
    trigger_terms={"research", "agent", "evidence"},
    eligible_categories={"technical_research"},
    steps=[
        SkillStep(order=1, instruction="Run ten broad searches before synthesizing."),
    ],
    invariants=["Do not fabricate citations."],
    source_memory_ids=["memory_bad_1"],
    source_run_ids=["run_failed_001"],
)
registry.register(bad_skill)
registry.begin_shadow(bad_skill.id)
bad_evaluation = registry.evaluate(bad_skill.id, benchmark_cases)
rejected_skill = registry.decide_promotion(bad_skill.id)

print("[V7][GOOD] Quality delta:", good_evaluation.quality_delta)
print("[V7][GOOD] Tool-call reduction:", good_evaluation.tool_call_delta)
print("[V7][GOOD] Status:", promoted_skill.status.value)
print("[V7][BAD] Policy violations:", bad_evaluation.policy_violations)
print("[V7][BAD] Status:", rejected_skill.status.value)

[V7][GOOD] Quality delta: 0.18
[V7][GOOD] Tool-call reduction: 2.0
[V7][GOOD] Status: active
[V7][BAD] Policy violations: 3
[V7][BAD] Status: rejected


## 7. Held-Out Reuse and Safe Versioning

The promoted Skill is now tested on a new technical research task. An ineligible breaking-news task receives no Skill.

We also register a V2 revision. It remains a candidate and cannot shadow or replace the active V1 until it completes the same lifecycle. This prevents silent self-modification.

In [7]:
held_out_research = BenchmarkCase(
    id="held_out_research",
    category="technical_research",
    request="Research evidence-backed agent checkpoint and memory patterns.",
    required_evidence_lanes={"academic", "official"},
)
held_out_news = BenchmarkCase(
    id="held_out_news",
    category="breaking_news",
    request="Report breaking sports news from the last ten minutes.",
    required_evidence_lanes={"fresh_news"},
)

selected_research_skill = registry.retrieve(held_out_research)
selected_news_skill = registry.retrieve(held_out_news)
held_out_result = execute_case(held_out_research, selected_research_skill)

revision_v2 = promoted_skill.model_copy(
    deep=True,
    update={
        "id": "academic_first_research_v2",
        "version": 2,
        "status": SkillStatus.CANDIDATE,
        "parent_skill_id": promoted_skill.id,
        "source_run_ids": promoted_skill.source_run_ids + ["run_held_out_001"],
    },
)
registry.register(revision_v2)
still_active = registry.retrieve(held_out_research)

print("[V7][HELD_OUT] Selected research Skill:", selected_research_skill.id)
print("[V7][HELD_OUT] Selected news Skill:", selected_news_skill)
print("[V7][HELD_OUT] Quality:", held_out_result.quality_score)
print("[V7][VERSION] Untested revision status:", revision_v2.status.value)
print("[V7][VERSION] Active version remains:", still_active.id)

[V7][HELD_OUT] Selected research Skill: academic_first_research_v1
[V7][HELD_OUT] Selected news Skill: None
[V7][HELD_OUT] Quality: 0.96
[V7][VERSION] Untested revision status: candidate
[V7][VERSION] Active version remains: academic_first_research_v1


## 8. End-to-End Evaluation

In [8]:
checks = {
    "candidate_has_memory_provenance": bool(
        candidate_skill.source_memory_ids and candidate_skill.source_run_ids
    ),
    "candidate_not_active_before_evaluation": candidate_skill.status
    == SkillStatus.CANDIDATE,
    "good_skill_improves_quality": good_evaluation.quality_delta
    >= registry.policy.minimum_quality_delta,
    "good_skill_reduces_calls": good_evaluation.tool_call_delta
    >= registry.policy.minimum_tool_call_reduction,
    "good_skill_has_no_policy_regression": good_evaluation.policy_violations == 0,
    "good_skill_not_misrouted": good_evaluation.ineligible_misroutes == 0,
    "good_skill_promoted": promoted_skill.status == SkillStatus.ACTIVE,
    "bad_skill_rejected": rejected_skill.status == SkillStatus.REJECTED,
    "held_out_research_uses_skill": held_out_result.skill_used,
    "held_out_news_uses_no_skill": selected_news_skill is None,
    "untested_revision_not_active": revision_v2.status == SkillStatus.CANDIDATE,
    "active_version_not_silently_replaced": still_active.id == promoted_skill.id,
}

for name, passed in checks.items():
    print(f"[V7][EVAL] {name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values()), [name for name, passed in checks.items() if not passed]
print(f"[V7][EVAL] Passed {sum(checks.values())}/{len(checks)} checks.")

[V7][EVAL] candidate_has_memory_provenance: PASS
[V7][EVAL] candidate_not_active_before_evaluation: PASS
[V7][EVAL] good_skill_improves_quality: PASS
[V7][EVAL] good_skill_reduces_calls: PASS
[V7][EVAL] good_skill_has_no_policy_regression: PASS
[V7][EVAL] good_skill_not_misrouted: PASS
[V7][EVAL] good_skill_promoted: PASS
[V7][EVAL] bad_skill_rejected: PASS
[V7][EVAL] held_out_research_uses_skill: PASS
[V7][EVAL] held_out_news_uses_no_skill: PASS
[V7][EVAL] untested_revision_not_active: PASS
[V7][EVAL] active_version_not_silently_replaced: PASS
[V7][EVAL] Passed 12/12 checks.


## 9. Embedded Skill-Learning Report

In [9]:
report_payload = {
    "good_skill": promoted_skill.model_dump(mode="json"),
    "good_evaluation": good_evaluation.model_dump(mode="json"),
    "bad_evaluation": bad_evaluation.model_dump(mode="json"),
    "held_out_result": held_out_result.model_dump(mode="json"),
    "active_skill_after_revision": still_active.id,
    "checks": checks,
}

fallback_report = (
    "# V7 Evaluation-Gated Skill Learning\n\n"
    "## Outcome\n\n"
    f"The candidate Skill improved mean quality by {good_evaluation.quality_delta:.2f} "
    f"and reduced mean tool calls by {good_evaluation.tool_call_delta:.1f}. It passed "
    "shadow evaluation and became active. A deliberately wasteful Skill was rejected "
    f"after {bad_evaluation.policy_violations} policy violation.\n\n"
    "## Safety Boundary\n\n"
    "Memory may propose procedures, but only the registry can activate a version after "
    "evaluation. An untested V2 revision remained a candidate while V1 continued serving "
    "eligible tasks. The breaking-news case received no research Skill.\n\n"
    "## Interpretation\n\n"
    "V7 demonstrates test-time procedural learning without model fine-tuning or unbounded "
    "self-modification. The system learns by promoting inspectable, reusable policies under "
    "a deterministic quality and safety gate."
)

live_report_used = False
final_report = fallback_report
if USE_LIVE_MODEL and client is not None:
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            instructions=(
                "Write a concise Markdown engineering report from the V7 evaluation JSON. "
                "Explain how procedural memory differs from a promoted Skill, why shadow "
                "evaluation prevents unsafe self-modification, the measured A/B result, "
                "and the limits of this simulation. Output only the report. Do not offer "
                "follow-up help."
            ),
            input=json.dumps(report_payload, indent=2),
        )
        if response.output_text.strip():
            final_report = response.output_text.strip()
            live_report_used = True
    except Exception as exc:
        print(f"[V7][REPORT] Live synthesis unavailable; using deterministic report: {type(exc).__name__}")

print(f"[V7][REPORT] Live model used: {live_report_used}")
print(f"[V7][REPORT] Characters: {len(final_report)}")
display(Markdown(final_report))

[V7][REPORT] Live model used: True
[V7][REPORT] Characters: 4917


V7 Engineering Report — Academic First Research (v1)

Executive summary
- The promoted skill academic_first_research_v1 passed shadow evaluation and reached production-promote status.
- A/B comparison favored the promoted skill: baseline quality 0.78 vs. candidate quality 0.96 (delta +0.18); baseline tool calls 4.0 vs. candidate 2.0 (reduced by 2 calls).
- Policy violations remained at 0; heuristic checks verified proper provenance, routing, and non-regression.
- Held-out evaluation confirms generalization: held-out result quality 0.96 with no policy violations, using the promoted skill.
- Bad evaluation example (exhaustive_search_v1) demonstrates risk when tool usage increases and policy constraints are violated.

Procedural memory vs promoted Skill
- Procedural memory
  - Operates as ephemeral, task-specific behavior within the current session.
  - Lacks versioning, explicit provenance, and formal safeguards; changes are not independently auditable or reuse-ready across contexts.
- Promoted Skill (academic_first_research_v1)
  - Versioned capability with defined provenance, invariants (no fabricated citations, no bypassing source provenance, respect tool-call budgets).
  - Memory of evidence sources is traceable (memory_1, memory_3; run_001) and subject to formal evaluation before promotion.
  - When promoted, it becomes a certified module with documented behavior and governance, reducing uncontrolled self-modification risk.

Shadow evaluation and unsafe self-modification prevention
- Shadow evaluation runs the candidate in a parallel, isolated environment (not active in live skill memory) to assess quality, evidence handling, and policy compliance without altering production pathways.
- Safeguards observed in this evaluation:
  - Active_version_not_silently_replaced and untested_revision_not_active ensure no unvetted changes affect live behavior.
  - Held-out cases (held_out_research) demonstrate how the skill performs on unseen data without influencing active operation.
  - News/held-out streams verify whether the candidate uses the skill or remains inert in contexts where no skill should drive behavior.
- Result: shadow evaluation prevented unsafe self-modification by isolating the candidate’s behavior and requiring formal promotion criteria before live activation.

Measured A/B result
- Evaluation scope
  - Relevant cases: 3
  - Held-out/result cases: 1 (held_out_research) plus evidence-collection cases
- Baseline vs. candidate
  - Baseline quality: 0.78
  - Candidate quality: 0.96
  - Quality delta: +0.18
  - Baseline tool calls: 4.0
  - Candidate tool calls: 2.0
  - Tool-call delta: -2.0 (reduction in tool usage)
  - Evidence coverage: all evaluated cases achieved 1.0
  - Policy violations: 0 in both baseline and candidate
- Case-level results (selected highlights)
  - case_1: tool_calls 2, quality_score 0.96, evidence_coverage 1.0, policy_violations 0
  - case_2: tool_calls 2, quality_score 0.96, evidence_coverage 1.0, policy_violations 0
  - case_3: tool_calls 2, quality_score 0.96, evidence_coverage 1.0, policy_violations 0
  - case_4_news: skill_used false; tool_calls 2; quality_score 0.91; evidence_coverage 1.0; policy_violations 0
- Held-out result
  - case_id: held_out_research
  - skill_id: academic_first_research_v1
  - tool_calls: 2
  - quality_score: 0.96
  - evidence_coverage: 1.0
  - policy_violations: 0
- Overall verdict
  - Passed: true
  - Active skill after revision: academic_first_research_v1

Limits of this simulation
- Scope and breadth
  - Evaluation covers 3 primary evidence cases plus a held-out test; not a full-scale production rollout.
- Representativeness
  - Results reflect controlled, synthetic test conditions; real-world data distributions and latency profiles may differ.
- Generalization
  - Held-out and news tests validate generalization to unseen inputs, but long-tail scenarios remain untested.
- Tooling and provenance
  - The evaluation enforces provenance and budget constraints; potential edge cases in untested toolchains or external docs are not captured.
- Temporal dynamics
  - The simulation does not model evolving external standards or evolving documentation beyond the evaluation window.

Key data highlights
- Good skill (academic_first_research_v1)
  - Status: active; promoted after evaluation
  - Cases evaluated: 3 (with 3 tool-usage-heavy cases), plus held-out case
  - Evidence sources: memory_1, memory_3; run_001
- Metrics at a glance
  - Baseline quality: 0.78
  - Candidate quality: 0.96
  - Quality delta: +0.18
  - Baseline tool calls: 4.0
  - Candidate tool calls: 2.0
  - Policy violations: 0 (baseline and candidate)
  - Held-out result: 0.96 quality, 1.0 evidence coverage, 0 policy violations
- Bad evaluation contrast
  - Exhaustive search (v1) showed deterioration in safety controls: 3 policy violations across 3 cases, 10 tool calls per case, failed overall

End of report

## 10. V7 Completion Summary

V7 adds a safe learning lifecycle on top of V6 memory:

`memory -> distillation -> candidate -> shadow -> eval gate -> promotion -> selective reuse -> version proposal`

The result is an agent that can improve its reusable procedures without confusing plausible text with proven behavior. V8 will decide when to use different model and tool capabilities under quality, cost, latency, and risk constraints.